# Modelling — Klasifikasi Teks Komentar Program KSP

## 1. Setup & Load Data

In [ ]:
import os
import re
import time
import warnings

import joblib
import numpy as np
import pandas as pd
from scipy.stats import randint, uniform
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    cross_val_score,
    train_test_split,
)
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.svm import SVC

warnings.filterwarnings("ignore")

# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_PATH = r"../data/data_labelling/data_labelled.xlsx"
SHEET_NAME = "Data Komentar"
MODEL_DIR = "model/"
TEXT_COL = "Isi Komentar"
TARGET_ISU = "Kategori Isu"
TARGET_SENT = "Sentimen"
RANDOM_STATE = 42
TEST_SIZE = 0.2

os.makedirs(MODEL_DIR, exist_ok=True)

# ── Load & filter valid rows ────────────────────────────────────────────────────
df_raw = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)
df = df_raw[df_raw[TARGET_ISU].notna() & df_raw[TARGET_SENT].notna()].reset_index(
    drop=True
)

print(f"Total valid rows : {len(df)}")
print(f"Kategori Isu     : {df[TARGET_ISU].nunique()} kelas")
print(f"Sentimen         : {df[TARGET_SENT].nunique()} kelas")
df[[TEXT_COL, TARGET_ISU, TARGET_SENT]].head

Total valid rows : 473
Kategori Isu     : 8 kelas
Sentimen         : 3 kelas


<bound method NDFrame.head of                                           Isi Komentar  \
0    Aamiin ya Allah 🤲🤲🤲\nPalagi kami tau ngk mampu...   
1                                    Mawaddah aamiin 🤲   
2                                             Amin dik   
3          Sate ku dadi tau pintar bau kena kartu nan.   
4        Pang ksb ta saser kk\nKaleng TK sampai kuliah   
..                                                 ...   
468                                             Mantap   
469                     Paling dekena grub nya Bae sii   
470                              Gimana cara daftarnya   
471  Tak ada sosialisasi yg lengkap bgm siswa menda...   
472   Mantap pak..yg msh kita tunggu lapangan kerja...   

                          Kategori Isu Sentimen  
0                          Doa/Harapan  Positif  
1                          Doa/Harapan  Positif  
2                          Doa/Harapan  Positif  
3                          Doa/Harapan   Netral  
4      Interaksi Sosial

## 2. Preprocessing Teks

In [ ]:
try:
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
    from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

    _stemmer = StemmerFactory().create_stemmer()
    _stop_factory = StopWordRemoverFactory()
    _stopwords = set(_stop_factory.get_stop_words())
    SASTRAWI_AVAILABLE = True
except ImportError:
    SASTRAWI_AVAILABLE = False
    _stemmer = None
    _stopwords = set()

_BASIC_STOPWORDS = {
    "yang",
    "dan",
    "di",
    "ke",
    "dari",
    "ini",
    "itu",
    "ada",
    "adalah",
    "dengan",
    "untuk",
    "tidak",
    "juga",
    "akan",
    "pada",
    "saya",
    "kami",
    "kita",
    "mereka",
    "sudah",
    "bisa",
    "lebih",
    "jika",
    "tapi",
    "atau",
    "karena",
    "agar",
    "maka",
    "bahwa",
    "oleh",
    "bagi",
    "kamu",
    "anda",
    "iya",
    "ya",
    "yg",
    "nya",
    "pun",
    "lah",
    "kah",
}
if not SASTRAWI_AVAILABLE:
    _stopwords = _BASIC_STOPWORDS


def preprocess_text(text: str) -> str:
    """Reusable preprocessing pipeline untuk teks Bahasa Indonesia."""
    if not isinstance(text, str) or not text.strip():
        return ""
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[@#]\w+", " ", text)
    text = text.encode("ascii", errors="ignore").decode("ascii")
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = [w for w in text.split() if w not in _stopwords and len(w) > 1]
    if SASTRAWI_AVAILABLE and _stemmer:
        tokens = [_stemmer.stem(w) for w in tokens]
    return " ".join(tokens)


print(f"Sastrawi tersedia : {SASTRAWI_AVAILABLE}")
sample = df[TEXT_COL].iloc[0]
print(f"Before : {repr(sample)}")
print(f"After  : {repr(preprocess_text(sample))}")

Sastrawi tersedia : True
Before : 'Aamiin ya Allah 🤲🤲🤲\nPalagi kami tau ngk mampu roa gama na pee'
After  : 'aamiin allah palagi tau ngk mampu roa gama na pee'


## 3. Feature Engineering — TF-IDF

In [ ]:
df["text_clean"] = df[TEXT_COL].apply(preprocess_text)

TFIDF_PARAMS = dict(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
)

tfidf_preview = TfidfVectorizer(**TFIDF_PARAMS)
X_preview = tfidf_preview.fit_transform(df["text_clean"])
print(f"Vocabulary size : {len(tfidf_preview.vocabulary_)} terms")
print(f"Matrix shape    : {X_preview.shape}")

Vocabulary size : 963 terms
Matrix shape    : (473, 963)


## 4. Train/Test Split

In [ ]:
X = df["text_clean"]
y_isu = df[TARGET_ISU]
y_sent = df[TARGET_SENT]

(
    X_train,
    X_test,
    y_isu_train,
    y_isu_test,
    y_sent_train,
    y_sent_test,
) = train_test_split(
    X,
    y_isu,
    y_sent,
    test_size=TEST_SIZE,
    stratify=y_isu,
    random_state=RANDOM_STATE,
)

print(f"Train : {len(X_train)} | Test : {len(X_test)}")
print("Distribusi Kategori Isu (train):")
print(y_isu_train.value_counts().to_string())

Train : 378 | Test : 95
Distribusi Kategori Isu (train):
Kategori Isu
Pertanyaan Prosedur & Persyaratan    81
Kritik/Keluhan                       72
Interaksi Sosial Antar Warganet      61
Doa/Harapan                          49
Klarifikasi Cakupan Program          33
Apresiasi/Dukungan                   32
Tidak Relevan/Tidak Jelas            29
Permohonan Bantuan Pribadi           21


## 5. Pemilihan Jumlah Fold CV Optimal

In [7]:
_baseline_pipe = make_pipeline(
    TfidfVectorizer(**TFIDF_PARAMS),
    MultinomialNB(),
)

k_candidates = [3, 5, 10, 15, 20, 25, 30]
cv_results = []

for k in k_candidates:
    t0 = time.perf_counter()
    scores = cross_val_score(
        _baseline_pipe,
        X_train,
        y_isu_train,
        cv=k,
        scoring="f1_macro",
        n_jobs=-1,
    )
    elapsed = time.perf_counter() - t0
    cv_results.append(
        {"k": k, "mean_f1": scores.mean(), "std_f1": scores.std(), "time_s": elapsed}
    )

cv_df = pd.DataFrame(cv_results).set_index("k")
print(cv_df.round(4).to_string())

_std_max_k = cv_df.loc[k_candidates[-1], "std_f1"]
_threshold = _std_max_k * 1.5
eligible = cv_df[cv_df["std_f1"] <= _threshold]
BEST_CV = int(eligible["time_s"].idxmin())

print(f"\nBEST_CV = {BEST_CV}")

    mean_f1  std_f1  time_s
k                          
3    0.3383  0.0032  0.0384
5    0.3561  0.0496  0.0360
10   0.3583  0.0642  0.0565
15   0.3530  0.0778  0.0815
20   0.3519  0.0965  0.0965
25   0.3410  0.0897  0.1376
30   0.3459  0.1300  0.1497

BEST_CV = 5


## 6. Hyperparameter Tuning — RandomizedSearchCV

In [8]:
N_ITER = 20

ALGO_CONFIGS = [
    (
        "SVM",
        Pipeline(
            [
                ("tfidf", TfidfVectorizer(**TFIDF_PARAMS)),
                ("clf", SVC(probability=True, random_state=RANDOM_STATE)),
            ]
        ),
        {
            "tfidf__max_features": [None, 5000, 10000],
            "clf__C": uniform(0.1, 10),
            "clf__kernel": ["linear", "rbf"],
            "clf__gamma": ["scale", "auto"],
        },
    ),
    (
        "Naive Bayes",
        Pipeline(
            [
                ("tfidf", TfidfVectorizer(**TFIDF_PARAMS)),
                ("clf", MultinomialNB()),
            ]
        ),
        {
            "tfidf__max_features": [None, 5000, 10000],
            "clf__alpha": uniform(0.01, 2.0),
        },
    ),
    (
        "KNN",
        Pipeline(
            [
                ("tfidf", TfidfVectorizer(**TFIDF_PARAMS)),
                ("clf", KNeighborsClassifier()),
            ]
        ),
        {
            "tfidf__max_features": [None, 5000, 10000],
            "clf__n_neighbors": randint(3, 20),
            "clf__weights": ["uniform", "distance"],
            "clf__metric": ["cosine", "euclidean"],
        },
    ),
    (
        "Random Forest",
        Pipeline(
            [
                ("tfidf", TfidfVectorizer(**TFIDF_PARAMS)),
                ("clf", RandomForestClassifier(random_state=RANDOM_STATE)),
            ]
        ),
        {
            "tfidf__max_features": [None, 5000, 10000],
            "clf__n_estimators": randint(100, 400),
            "clf__max_depth": [None, 10, 20, 30],
            "clf__min_samples_split": randint(2, 10),
        },
    ),
]


def tune_algorithms(X_tr, y_tr):
    """Run RandomizedSearchCV for all algorithms; return list of result dicts."""
    results = []
    for name, pipe, param_dist in ALGO_CONFIGS:
        print(f"  Tuning {name} ...", end=" ", flush=True)
        t0 = time.perf_counter()
        search = RandomizedSearchCV(
            pipe,
            param_distributions=param_dist,
            n_iter=N_ITER,
            cv=BEST_CV,
            scoring="f1_macro",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            refit=True,
        )
        search.fit(X_tr, y_tr)
        elapsed = time.perf_counter() - t0
        print(f"done ({elapsed:.1f}s) | best CV f1_macro={search.best_score_:.4f}")
        results.append(
            {
                "algorithm": name,
                "best_params": search.best_params_,
                "best_estimator": search.best_estimator_,
                "cv_f1_macro": search.best_score_,
                "tuning_time_s": elapsed,
            }
        )
    return results


print("=== Tuning — Kategori Isu ===")
results_isu = tune_algorithms(X_train, y_isu_train)

print("\n=== Tuning — Sentimen ===")
results_sent = tune_algorithms(X_train, y_sent_train)

=== Tuning — Kategori Isu ===
  Tuning SVM ... done (2.0s) | best CV f1_macro=0.4843
  Tuning Naive Bayes ... done (0.5s) | best CV f1_macro=0.4478
  Tuning KNN ... done (0.7s) | best CV f1_macro=0.4457
  Tuning Random Forest ... done (12.0s) | best CV f1_macro=0.4451

=== Tuning — Sentimen ===
  Tuning SVM ... done (1.7s) | best CV f1_macro=0.6619
  Tuning Naive Bayes ... done (0.7s) | best CV f1_macro=0.5996
  Tuning KNN ... done (0.7s) | best CV f1_macro=0.5933
  Tuning Random Forest ... done (15.0s) | best CV f1_macro=0.5425


## 7. Evaluasi Model pada Data Test

In [9]:
def evaluate_on_test(results, X_te, y_te):
    """Evaluate each best_estimator on test set; augment result dicts in-place."""
    for r in results:
        y_pred = r["best_estimator"].predict(X_te)
        r["accuracy"] = accuracy_score(y_te, y_pred)
        r["precision_macro"] = precision_score(
            y_te, y_pred, average="macro", zero_division=0
        )
        r["recall_macro"] = recall_score(y_te, y_pred, average="macro", zero_division=0)
        r["f1_macro"] = f1_score(y_te, y_pred, average="macro", zero_division=0)
        r["confusion_matrix"] = confusion_matrix(y_te, y_pred)
    return results


results_isu = evaluate_on_test(results_isu, X_test, y_isu_test)
results_sent = evaluate_on_test(results_sent, X_test, y_sent_test)

for target_name, results in [(TARGET_ISU, results_isu), (TARGET_SENT, results_sent)]:
    print(f"\n{'='*60}")
    print(f"Target: {target_name}")
    for r in results:
        print(f"\n  {r['algorithm']}")
        print(f"    Accuracy  : {r['accuracy']:.4f}")
        print(f"    Precision : {r['precision_macro']:.4f}")
        print(f"    Recall    : {r['recall_macro']:.4f}")
        print(f"    F1 Macro  : {r['f1_macro']:.4f}")
        print(f"    Confusion Matrix:\n{r['confusion_matrix']}")


Target: Kategori Isu

  SVM
    Accuracy  : 0.6947
    Precision : 0.7303
    Recall    : 0.6417
    F1 Macro  : 0.6594
    Confusion Matrix:
[[ 5  2  1  0  0  0  0  0]
 [ 0  9  0  0  1  1  1  0]
 [ 0  0 11  0  3  1  0  0]
 [ 0  0  1  6  0  1  1  0]
 [ 0  0  2  1 15  0  0  0]
 [ 0  1  1  1  0  2  0  0]
 [ 0  1  2  1  1  0 15  0]
 [ 0  0  4  0  0  0  1  3]]

  Naive Bayes
    Accuracy  : 0.6105
    Precision : 0.6472
    Recall    : 0.5726
    F1 Macro  : 0.5707
    Confusion Matrix:
[[ 3  3  1  0  0  0  1  0]
 [ 0  9  0  0  1  1  1  0]
 [ 0  0  7  3  2  0  3  0]
 [ 0  0  0  6  2  1  0  0]
 [ 0  2  0  1 13  0  2  0]
 [ 0  0  0  0  0  3  2  0]
 [ 1  1  1  2  0  0 15  0]
 [ 1  0  1  0  0  1  3  2]]

  KNN
    Accuracy  : 0.5579
    Precision : 0.5931
    Recall    : 0.5201
    F1 Macro  : 0.5228
    Confusion Matrix:
[[ 3  4  1  0  0  0  0  0]
 [ 0 10  0  0  0  1  1  0]
 [ 0  4  4  2  3  0  2  0]
 [ 0  0  2  5  0  1  1  0]
 [ 0  2  1  1 10  1  3  0]
 [ 0  1  0  0  0  2  1  1]
 [ 0  1  1 

## 8. Tabel Ringkasan Perbandingan Model

In [10]:
def build_summary_table(results):
    """Return DataFrame sorted by F1 Macro descending."""
    rows = []
    for r in results:
        rows.append(
            {
                "Algorithm": r["algorithm"],
                "Best Params": str(r["best_params"]),
                "Accuracy": round(r["accuracy"], 4),
                "Precision Macro": round(r["precision_macro"], 4),
                "Recall Macro": round(r["recall_macro"], 4),
                "F1 Macro": round(r["f1_macro"], 4),
                "Tuning Time (s)": round(r["tuning_time_s"], 2),
            }
        )
    return (
        pd.DataFrame(rows)
        .sort_values("F1 Macro", ascending=False)
        .reset_index(drop=True)
    )


summary_isu = build_summary_table(results_isu)
summary_sent = build_summary_table(results_sent)

print("=== Kategori Isu ===")
display(summary_isu)

print("\n=== Sentimen ===")
display(summary_sent)

=== Kategori Isu ===


,Algorithm,Best Params,Accuracy,Precision Macro,Recall Macro,F1 Macro,Tuning Time (s)
0,SVM,"{'clf__C': np.float64(1.6599452033620266), 'cl...",0.6947,0.7303,0.6417,0.6594,2.02
1,Random Forest,"{'clf__max_depth': None, 'clf__min_samples_spl...",0.6000,0.6311,0.5590,0.5749,12.04
2,Naive Bayes,{'clf__alpha': np.float64(0.051168988591604896...,0.6105,0.6472,0.5726,0.5707,0.48
3,KNN,"{'clf__metric': 'cosine', 'clf__n_neighbors': ...",0.5579,0.5931,0.5201,0.5228,0.72



=== Sentimen ===


,Algorithm,Best Params,Accuracy,Precision Macro,Recall Macro,F1 Macro,Tuning Time (s)
0,Naive Bayes,{'clf__alpha': np.float64(0.051168988591604896...,0.7368,0.6623,0.6089,0.6293,0.68
1,KNN,"{'clf__metric': 'cosine', 'clf__n_neighbors': ...",0.7263,0.6661,0.5918,0.6159,0.75
2,SVM,"{'clf__C': np.float64(1.6599452033620266), 'cl...",0.7368,0.6430,0.5575,0.5823,1.73
3,Random Forest,"{'clf__max_depth': 30, 'clf__min_samples_split...",0.6947,0.6980,0.4726,0.4945,14.98


## 9. Pilih Model Terbaik Secara Otomatis

In [11]:
def select_best(results):
    """Return result dict with highest F1 Macro on test set."""
    return max(results, key=lambda r: r["f1_macro"])


best_isu = select_best(results_isu)
best_sent = select_best(results_sent)

print(
    f"Best model — Kategori Isu : {best_isu['algorithm']}  (F1={best_isu['f1_macro']:.4f})"
)
print(
    f"Best model — Sentimen     : {best_sent['algorithm']}  (F1={best_sent['f1_macro']:.4f})"
)

Best model — Kategori Isu : SVM  (F1=0.6594)
Best model — Sentimen     : Naive Bayes  (F1=0.6293)


## 10. Simpan Model & Tabel Ringkasan

In [12]:
def slugify(name: str) -> str:
    return name.lower().replace(" ", "_")


model_isu_path = os.path.join(
    MODEL_DIR, f"best_model_kategori_isu_{slugify(best_isu['algorithm'])}.joblib"
)
model_sent_path = os.path.join(
    MODEL_DIR, f"best_model_sentimen_{slugify(best_sent['algorithm'])}.joblib"
)
summary_isu_csv = os.path.join(MODEL_DIR, "summary_kategori_isu.csv")
summary_sent_csv = os.path.join(MODEL_DIR, "summary_sentimen.csv")

joblib.dump(best_isu["best_estimator"], model_isu_path)
joblib.dump(best_sent["best_estimator"], model_sent_path)
summary_isu.to_csv(summary_isu_csv, index=False)
summary_sent.to_csv(summary_sent_csv, index=False)

print(f"Saved: {model_isu_path}")
print(f"Saved: {model_sent_path}")
print(f"Saved: {summary_isu_csv}")
print(f"Saved: {summary_sent_csv}")

Saved: model/best_model_kategori_isu_svm.joblib
Saved: model/best_model_sentimen_naive_bayes.joblib
Saved: model/summary_kategori_isu.csv
Saved: model/summary_sentimen.csv


## 11. Fungsi Prediksi Data Baru

In [13]:
def predict_new_comments(df_baru: pd.DataFrame) -> pd.DataFrame:
    """
    Prediksi Kategori Isu dan Sentimen untuk komentar baru.

    Parameters
    ----------
    df_baru : pd.DataFrame
        DataFrame dengan setidaknya kolom 'Isi Komentar'.
        Kolom label tidak diperlukan.

    Returns
    -------
    pd.DataFrame
        DataFrame asli + kolom pred_kategori_isu, conf_kategori_isu,
        pred_sentimen, conf_sentimen.
    """

    def _find_model(prefix):
        candidates = [
            f
            for f in os.listdir(MODEL_DIR)
            if f.startswith(prefix) and f.endswith(".joblib")
        ]
        if not candidates:
            raise FileNotFoundError(
                f"Tidak ditemukan file model dengan prefix '{prefix}' di '{MODEL_DIR}'. "
                "Jalankan sel 10 terlebih dahulu."
            )
        return os.path.join(MODEL_DIR, sorted(candidates)[-1])

    pipe_isu = joblib.load(_find_model("best_model_kategori_isu"))
    pipe_sent = joblib.load(_find_model("best_model_sentimen"))

    result = df_baru.copy()
    texts = result[TEXT_COL].apply(preprocess_text)

    result["pred_kategori_isu"] = pipe_isu.predict(texts)
    if hasattr(pipe_isu, "predict_proba"):
        result["conf_kategori_isu"] = pipe_isu.predict_proba(texts).max(axis=1).round(4)
    else:
        result["conf_kategori_isu"] = np.nan

    result["pred_sentimen"] = pipe_sent.predict(texts)
    if hasattr(pipe_sent, "predict_proba"):
        result["conf_sentimen"] = pipe_sent.predict_proba(texts).max(axis=1).round(4)
    else:
        result["conf_sentimen"] = np.nan

    return result


print("Fungsi predict_new_comments siap digunakan.")

Fungsi predict_new_comments siap digunakan.


## 12. Simulasi Prediksi Data Baru (End-to-End Verification)

In [24]:
# Ambil 5 baris dari data test sebagai simulasi komentar baru (tanpa label)
df_simulasi = df.loc[
    X_test.index[70:80],
    ["No", "Tanggal", "Platform/Sumber", "Nama Akun", TEXT_COL],
].reset_index(drop=True)

df_hasil = predict_new_comments(df_simulasi)

cols_show = [
    TEXT_COL,
    "pred_kategori_isu",
    "conf_kategori_isu",
    "pred_sentimen",
    "conf_sentimen",
]
display(df_hasil[cols_show])

print("\nLabel asli:")
display(
    df.loc[X_test.index[70:80], [TEXT_COL, TARGET_ISU, TARGET_SENT]].reset_index(drop=True)
)

,Isi Komentar,pred_kategori_isu,conf_kategori_isu,pred_sentimen,conf_sentimen
0,Mantap,Apresiasi/Dukungan,0.5346,Netral,0.6930
1,"RahmadSueb usul saja mas, tidak perlu dgn mend...",Klarifikasi Cakupan Program,0.6514,Netral,0.8065
2,Usul sendiri dan cukup pemerintah memberikan r...,Klarifikasi Cakupan Program,0.4807,Netral,0.9058
3,"Harus semuanya kenah sekarang,karna udah penda...",Interaksi Sosial Antar Warganet,0.2136,Negatif,0.5062
4,Luar biasa pak bupati Syarafuddin Jarot👍👍👍,Apresiasi/Dukungan,0.6074,Positif,0.9893
5,Ishak Syahril betul sekali pak👍🏻,Interaksi Sosial Antar Warganet,0.2293,Netral,0.6516
6,Aamiin,Doa/Harapan,0.7251,Netral,0.8953
7,STAI NW Samawa terlewat pak\n\nMohon infonya j...,Klarifikasi Cakupan Program,0.2161,Netral,0.9648
8,Aamiin,Doa/Harapan,0.7251,Netral,0.8953
9,Dwi Rahayu MJ bagaimana caranya bu,Pertanyaan Prosedur & Persyaratan,0.2653,Positif,0.6446



Label asli:


,Isi Komentar,Kategori Isu,Sentimen
0,Mantap,Apresiasi/Dukungan,Positif
1,"RahmadSueb usul saja mas, tidak perlu dgn mend...",Klarifikasi Cakupan Program,Netral
2,Usul sendiri dan cukup pemerintah memberikan r...,Klarifikasi Cakupan Program,Netral
3,"Harus semuanya kenah sekarang,karna udah penda...",Klarifikasi Cakupan Program,Netral
4,Luar biasa pak bupati Syarafuddin Jarot👍👍👍,Apresiasi/Dukungan,Positif
5,Ishak Syahril betul sekali pak👍🏻,Interaksi Sosial Antar Warganet,Netral
6,Aamiin,Doa/Harapan,Netral
7,STAI NW Samawa terlewat pak\n\nMohon infonya j...,Permohonan Bantuan Pribadi,Netral
8,Aamiin,Doa/Harapan,Netral
9,Dwi Rahayu MJ bagaimana caranya bu,Pertanyaan Prosedur & Persyaratan,Netral
